# Faz 1 — Veri İnceleme

Bu notebook'ta amacımız **hiçbir kolon ismini veya veri formatını varsaymadan**, Steam Games Dataset'in (FronkonGames, Kaggle) ham halini inceleyip anlamak.

Veri setinin iki versiyonu var: `games.csv` ve `games.json`. Önce ikisini karşılaştırıp hangisini kullanacağımıza karar vereceğiz.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

RAW_DIR = Path('../data/raw')

## 1. CSV vs JSON — hangi versiyonu kullanmalı?

Projenin dokümanında bilinen bir sorun belirtiliyor: CSV versiyonunda başlıkların kayması/birleşmesi (`DiscountDLC count` gibi) nedeniyle kolonlar bir kaymış olabilir. Önce bunu doğrulayalım.

In [ ]:
# CSV'nin header'ını ve ilk satırlarını (düşük bellek maliyetiyle) incele
csv_preview = pd.read_csv(RAW_DIR / 'games.csv', nrows=5)
print('CSV kolon sayısı:', csv_preview.shape[1])
csv_preview.columns.tolist()

In [ ]:
# İlk satırın değerlerini kolon adlarının yanına koyup kayma olup olmadığına bakalım
row0 = csv_preview.iloc[0]
for col in csv_preview.columns[:10]:
    val = str(row0[col])
    print(f"{col!r:28} -> {val[:50]}")

**Gözlem:** `AppID` kolonunun altında oyunun adı (`Black Dragon Mage Playtest`), `Name` kolonunun altında çıkış tarihi (`Aug 1, 2023`) var. Yani CSV'nin header'ı ile verisi **bir kolon kaymış**.

**Sebep:** CSV header satırında `DiscountDLC count` tek bir başlık olarak görünüyor, ama veride bu aslında iki ayrı alan (`Discount` ve `DLC count`). Yani her satırda 40 veri alanı var ama header'da 39 isim var. `pandas.read_csv`, header'dan bir fazla alanı gördüğünde ilk alanı (gerçek AppID) **index** olarak kullanıyor ve kalan 39 veri alanını 39 başlığa kaydırmalı eşliyor. Bu kayma dosyanın tamamında tutarlı şekilde devam ediyor; yani her kolonun gerçek değeri, bir sonraki ham veri alanına denk geliyor.

**Karar: CSV yerine JSON versiyonunu kullanacağız.**
- JSON'da her alan kendi adıyla ayrı ayrı geliyor (`name`, `release_date`, `discount`, `dlc_count`, ...) → kayma problemi yok.
- JSON'da CSV'de bulunmayan `detailed_description` ve `short_description` alanları da var.
- JSON'da liste tipi alanlar (`genres`, `categories`, `developers`, ...) zaten Python listesi olarak geliyor; CSV'de bunlar virgülle ayrılmış tek bir metin.
- Tek dezavantajı dosyanın daha büyük olması (948 MB) ve yüklemenin biraz daha yavaş olması (~9 sn, ~4.3 GB pik bellek) — 16 GB RAM'li bu makinede sorun değil.

In [ ]:
# JSON versiyonunu yükle (anahtar = app_id)
df = pd.read_json(RAW_DIR / 'games.json', orient='index')
df.index.name = 'app_id'
df = df.reset_index()
print(df.shape)
df.head(3)

## 2. Satır/Kolon Sayısı, Veri Tipleri ve Bellek Kullanımı

In [ ]:
print(f"Satır sayısı : {len(df):,}")
print(f"Kolon sayısı : {df.shape[1]}")
print(f"Bellek kullanımı (deep): {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
df.dtypes

In [ ]:
# JSON ve CSV satır sayılarını karşılaştır (CSV'yi tam yüklemeden, sadece satır sayarak)
with open(RAW_DIR / 'games.csv', encoding='utf-8') as f:
    csv_row_count = sum(1 for _ in f) - 1  # header hariç

print(f"CSV satır sayısı  : {csv_row_count:,}")
print(f"JSON satır sayısı : {len(df):,}")
print(f"Fark              : {len(df) - csv_row_count:,}")

**Not:** JSON versiyonu CSV'den daha fazla satır içeriyor. Bu, iki dosyanın farklı zamanlarda veya farklı filtrelerle üretilmiş olabileceğini gösteriyor. CSV'de bildiğimiz bir yapısal sorun olduğu için JSON'u esas alıyoruz; bu fark raporda bir sınırlılık olarak not edilecek.

Tekrarlı `app_id` olup olmadığını da kontrol edelim.

In [ ]:
print('Tekrarlı app_id sayısı:', df['app_id'].duplicated().sum())

## 3. Veri Sözlüğü: Eksik Değer Oranı, Benzersiz Değer Sayısı, Örnek Değer

**Önemli not:** Bu veri setinde eksik veriler çoğunlukla gerçek `NaN` olarak değil, **varsayılan/sentinel değerlerle** (boş string `""`, boş liste `[]`, boş sözlük `{}`, sayısal alanlarda `0`) temsil ediliyor. Aşağıdaki tablo teknik `NaN` oranını gösteriyor; "gerçek" eksiklik (örn. hiç inceleme almamış oyun) Faz 2'de alan bazında ayrıca ele alınacak.

In [ ]:
def _nunique_safe(series):
    """list/dict gibi hashlenemeyen tipler için güvenli nunique."""
    def to_hashable(v):
        if isinstance(v, list):
            return tuple(v)
        if isinstance(v, dict):
            return tuple(sorted(v.items()))
        return v
    return series.map(to_hashable).nunique(dropna=True)


def build_data_dictionary(frame):
    rows = []
    for col in frame.columns:
        s = frame[col]
        non_null = s.dropna()
        sample = non_null.iloc[0] if len(non_null) else None
        sample_str = str(sample)
        if len(sample_str) > 60:
            sample_str = sample_str[:60] + '...'
        rows.append({
            'kolon': col,
            'tip': str(s.dtype),
            'eksik_orani_nan': s.isna().mean(),
            'benzersiz_sayisi': _nunique_safe(s),
            'ornek_deger': sample_str,
        })
    return pd.DataFrame(rows)


data_dict = build_data_dictionary(df)
data_dict

## 4. Beklenen Önemli Kolonların Teyidi

Doküman, önemli kabul edilen kolonları isim vermeden (kavram olarak) listeliyor. Gerçek JSON alan adlarıyla eşleştirip teyit edelim.

In [ ]:
expected = {
    'Uygulama ID': 'app_id',
    'İsim': 'name',
    'Çıkış tarihi': 'release_date',
    'Tahmini sahip sayısı (aralık)': 'estimated_owners',
    'Eş zamanlı oyuncu zirvesi (Peak CCU)': 'peak_ccu',
    'Fiyat': 'price',
    'DLC sayısı': 'dlc_count',
    'Desteklenen diller': 'supported_languages',
    'Windows desteği': 'windows',
    'Mac desteği': 'mac',
    'Linux desteği': 'linux',
    'Metacritic puanı': 'metacritic_score',
    'Kullanıcı puanı': 'user_score',
    'Olumlu review sayısı': 'positive',
    'Olumsuz review sayısı': 'negative',
    'Başarım sayısı': 'achievements',
    'Ortalama oynama süresi': 'average_playtime_forever',
    'Medyan oynama süresi': 'median_playtime_forever',
    'Geliştirici': 'developers',
    'Yayıncı': 'publishers',
    'Kategoriler': 'categories',
    'Türler': 'genres',
    'Etiketler': 'tags',
}

for turkce, col in expected.items():
    var_mi = 'OK' if col in df.columns else 'YOK'
    print(f"[{var_mi}] {turkce:40} -> {col}")

print()
print('JSON\'da olup yukarıdaki listede olmayan ekstra kolonlar:')
print(sorted(set(df.columns) - set(expected.values())))

Beklenen tüm kolonlar JSON'da mevcut. Ekstra olarak açıklama metinleri (`detailed_description`, `short_description`, `about_the_game`), `discount`, `required_age`, görsel/URL alanları (`header_image`, `screenshots`, `movies`, `website`, `support_url`, `support_email`), `packages`, `notes`, `score_rank`, `recommendations`, `metacritic_url` bulunuyor.

## 5. Liste/Sözlük Tipi Alanlara Yakından Bakış

In [ ]:
sample_idx = 1
for col in ['supported_languages', 'full_audio_languages', 'developers', 'publishers',
            'categories', 'genres', 'tags', 'packages']:
    print(f"{col:22} ({type(df[col].iloc[sample_idx]).__name__}) -> {df[col].iloc[sample_idx]}")

**Önemli fark:** `genres`, `categories`, `developers`, `publishers`, `supported_languages` birer **liste**; ama `tags` bir **sözlük** (`{'Adventure': 27, 'Visual Novel': 19, ...}` — etiket adı → o etiketi kaç kullanıcının verdiği). Faz 2'de `tags` için listeye çevirirken bu sayıları da (opsiyonel ağırlık olarak) saklamayı düşünebiliriz; şimdilik sadece anahtarları (etiket adlarını) liste olarak çıkarmak yeterli.

## 6. Önemli Alanlarda Hızlı Dağılım Kontrolü

In [ ]:
print('--- estimated_owners en sık görülen değerler ---')
print(df['estimated_owners'].value_counts().head(10))
print()
print(f"'0 - 0' (muhtemelen hiç sahibi/oyuncusu olmayan) oranı: {(df['estimated_owners'] == '0 - 0').mean():.1%}")

In [ ]:
print('--- release_date format kontrolü ---')
# Beklenen format: "Mon D, YYYY" (örn. "Aug 1, 2023")
non_standard = df['release_date'][~df['release_date'].str.match(r'^[A-Za-z]{3} \d{1,2}, \d{4}$', na=False)]
print(f"Standart olmayan format sayısı: {len(non_standard)} / {len(df)}")
print(f"Boş/NaN sayısı: {df['release_date'].isna().sum()}")

In [ ]:
print('--- fiyat dağılımı ---')
print(df['price'].describe())
print(f"Ücretsiz oyun oranı (price == 0): {(df['price'] == 0).mean():.1%}")

In [ ]:
print('--- review durumu ---')
no_reviews = (df['positive'] == 0) & (df['negative'] == 0)
print(f"Hiç review'u olmayan oyun oranı: {no_reviews.mean():.1%}")

In [ ]:
print('--- isim / tür boşlukları ---')
print('Boş isim sayısı:', (df['name'].fillna('').str.strip() == '').sum())
print('Boş genres listesi sayısı:', df['genres'].apply(len).eq(0).sum())
print('Boş tags sözlüğü sayısı:', df['tags'].apply(len).eq(0).sum())

In [ ]:
print('--- Indie tanımı: genres vs tags tutarlılığı ---')
is_indie_genre = df['genres'].apply(lambda g: 'Indie' in g)
is_indie_tag = df['tags'].apply(lambda t: 'Indie' in t)
print(f"genres içinde 'Indie' olan oran : {is_indie_genre.mean():.1%}")
print(f"tags içinde 'Indie' olan oran   : {is_indie_tag.mean():.1%}")
print(f"İki tanım birbirinden farklı sonuç veren oyun sayısı: {(is_indie_genre != is_indie_tag).sum():,}")

## 7. Genel Gözlemler ve Faz 2 İçin Notlar

- **Veri kaynağı kararı:** CSV'de header/veri kayması olduğu için **JSON versiyonu** kullanılacak (bkz. Bölüm 1).
- Veri setinde **147.352 oyun** var, tekrarlı `app_id` yok.
- `release_date` tamamen `"Mon D, YYYY"` formatında ve hiç boş değil; ancak bazı tarihler gelecek tarihli (henüz çıkmamış/duyurulmuş oyunlar) olabilir — Faz 2'de kontrol edilip işaretlenecek.
- `estimated_owners`, beklendiği gibi `"X - Y"` aralık formatında bir metin; **%30'u `"0 - 0"`** yani sahiplik verisi pratikte yok — bu oyunlar muhtemelen hiç yayınlanmamış/oynanmamış.
- Oyunların **%43,7'sinin hiç review'u yok** (`positive == 0` ve `negative == 0`) — dokümanda belirtilen "çok sayıda oyunun review'u yok" durumu doğrulandı. Faz 2'de analiz iki seviyede yapılacak: tüm veri ve minimum review eşiğini geçen oyunlar.
- Oyunların **%19,8'i ücretsiz** (`price == 0`).
- `tags` alanı diğer liste alanlarından farklı olarak bir **sözlük** (etiket → oy sayısı); temizlemede özel olarak ele alınacak.
- **`is_indie` tanımı net değil:** `genres` içinde "Indie" oranı %66,8 iken `tags` içinde %33,0 — iki tanım **54.714 oyunda** birbirinden farklı sonuç veriyor. Faz 2'de hangi tanımın (genres, tags, ya da ikisinin birleşimi) kullanılacağına karar verilip gerekçelendirilecek.
- `name` alanında 1 oyun boş; `genres` alanı 8.437 oyunda boş liste; `tags` alanı 63.963 oyunda boş sözlük (çoğunlukla hiç review'u olmayan oyunlarla örtüşüyor olabilir).
- Teknik `NaN` oranı birçok kolonda düşük görünse de, bu veri setinde "eksiklik" genelde boş string/liste/sözlük veya `0` ile temsil ediliyor; Faz 2'de kolon bazında gerçek eksiklik tanımı yapılacak.